# Stage 3 Week 1: Baseline and First Poisoning Attack

**Research Question:** How do poisoning strategy and poisoning rate affect the reliability of classical spam classifiers, and how much performance can a manually implemented label-auditing defense recover?

---

## 1. Dataset Technical Description (UCI Spambase)
- **Source:** Hopkins, Reeber, Forman, Suermondt (Hewlett-Packard Labs, 1999)
- **Number of Instances:** 4,601 email messages
- **Features (57 continuous numerical attributes):**
  - **48 Word Frequencies (`word_freq_WORD`):** Percentage of words matching specific keywords (e.g., `make`, `address`, `all`, `3d`, `our`, `free`, `credit`, `money`).
  - **6 Character Frequencies (`char_freq_CHAR`):** Percentage of characters matching specific punctuation: `;`, `(`, `[`, `!`, `$`, `#`.
  - **3 Capital Run Length Attributes:** `capital_run_length_average`, `capital_run_length_longest`, and `capital_run_length_total`.
- **Target Encoding:** `1` = Spam (unsolicited commercial email), `0` = Non-spam (ham).
- **Class Distribution:** 2,788 Non-Spam (60.6%) vs. 1,813 Spam (39.4%).
- **Known Limitations:** Contains personal/HP-specific tokens (e.g., `george`, `650`, `hp`, `lab`) which act as strong indicators of non-spam for the donor, but would introduce bias in a general-purpose filter without blinding.

---

## 2. Threat Model (NIST AML Taxonomy)
- **Protected System:** Automated email security filter running classical ML classifiers.
- **Attacker:** External spammer/adversary attempting to bypass spam detection.
- **Attacker Access & Capabilities:** Training-time label control (can flip labels of a subset of training data without modifying feature values or accessing the test set).
- **Attacker Goal:** Integrity violation (evasion / reducing spam recall) and Availability violation (degrading filter accuracy).
- **Manipulated Asset:** Training labels ($y_{train}$).
- **Likely Harm:** High-risk spam/phishing delivered to user inboxes; legitimate messages misclassified.

In [ ]:
# Cell 1: Imports and environment setup
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))
from src.data import load_spambase, get_dataset_summary, split_data
from src.models import get_models
from src.metrics import evaluate_models
from src.attacks import random_label_flip
from src.visualization import (
    plot_class_distribution,
    plot_feature_comparison_box,
    plot_feature_comparison_hist,
    plot_confusion_matrices,
    plot_clean_vs_poisoned_metrics
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Environment initialized successfully.')

## 3. Dataset Loading & Health Checks

In [ ]:
# Cell 2: Load dataset and inspect summary
df = load_spambase(data_path='../data/spambase.data', names_path='../data/spambase.names')
summary = get_dataset_summary(df)

print(f"Rows: {summary['n_rows']}")
print(f"Columns: {summary['n_columns']} ({summary['n_features']} features + 1 target)")
print(f"Missing values: {summary['missing_values']}")
print(f"Duplicate rows: {summary['duplicate_rows']}")
print('Class distribution:', summary['class_distribution'])
print('Class proportions:', {k: f"{v*100:.2f}%" for k, v in summary['class_proportions'].items()})

## 4. Exploratory Data Visualizations (>= 3 Figures)

In [ ]:
# Cell 3: Visualization 1 - Class distribution
fig1, ax1 = plot_class_distribution(df['is_spam'], title='UCI Spambase Class Distribution')
plt.show()

In [ ]:
# Cell 4: Visualization 2 - Feature comparison (word_freq_free)
fig2, ax2 = plot_feature_comparison_box(df, feature_name='word_freq_free', title="Distribution of 'word_freq_free' by Email Class")
plt.show()

In [ ]:
# Cell 5: Visualization 3 - Feature comparison (char_freq_$ on log scale)
fig3, ax3 = plot_feature_comparison_hist(df, feature_name='char_freq_$', log_scale=True, title="Density Distribution of 'char_freq_$'")
plt.show()

## 5. Stratified 80/20 Train-Test Split (random_state=42)

In [ ]:
# Cell 6: Train-test split (test set must remain clean throughout all experiments)
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples (Non-spam: {(y_train==0).sum()}, Spam: {(y_train==1).sum()})")
print(f"Test set:     {X_test.shape[0]} samples (Non-spam: {(y_test==0).sum()}, Spam: {(y_test==1).sum()})")

# Fingerprint test set to prove strict isolation
test_feat_hash_init = pd.util.hash_pandas_object(X_test).sum()
test_label_hash_init = pd.util.hash_pandas_object(y_test).sum()

## 6. Clean Baseline Model Training and Evaluation

In [ ]:
# Cell 7: Train and evaluate clean models
clean_models = get_models(random_state=42)
for name, model in clean_models.items():
    model.fit(X_train, y_train)

clean_results, clean_cms = evaluate_models(clean_models, X_test, y_test)

print('=== Clean Baseline Results ===')
display(clean_results[['Model', 'Accuracy', 'Spam Precision', 'Spam Recall', 'Spam F1']])

# Plot 3 confusion matrices
fig_clean_cms, axes = plot_confusion_matrices(clean_cms, title_prefix='Clean Baseline (0% Poison)')
plt.show()

## 7. First Poisoning Attack: 5% Random Label Flipping

In [ ]:
# Cell 8: Apply 5% random label flipping to training labels
y_train_poisoned, flipped_mask = random_label_flip(y_train, poison_rate=0.05, random_state=42)

n_flipped = flipped_mask.sum()
print('Target Poison Rate: 5.0%')
print(f"Exact Flipped Training Labels: {n_flipped} / {len(y_train)} ({n_flipped / len(y_train) * 100:.2f}%)")

# Verification of Invariants
assert (y_train_poisoned != y_train).sum() == n_flipped, 'Flipped label count mismatch!'
assert pd.util.hash_pandas_object(X_test).sum() == test_feat_hash_init, 'Test features were corrupted!'
assert pd.util.hash_pandas_object(y_test).sum() == test_label_hash_init, 'Test labels were corrupted!'
print('✓ Invariant checks passed: Test set remains completely clean, only training labels were modified.')

## 8. Retrain on Poisoned Training Data & Performance Comparison

In [ ]:
# Cell 9: Retrain models with poisoned labels and evaluate on clean test set
poisoned_models = get_models(random_state=42)
for name, model in poisoned_models.items():
    model.fit(X_train, y_train_poisoned)

poisoned_results, poisoned_cms = evaluate_models(poisoned_models, X_test, y_test)

print('=== 5% Poisoned Results ===')
display(poisoned_results[['Model', 'Accuracy', 'Spam Precision', 'Spam Recall', 'Spam F1']])

# 4-panel comparison figure across Accuracy, Precision, Recall, and F1
fig_comp, axes_comp = plot_clean_vs_poisoned_metrics(clean_results, poisoned_results)
plt.show()

## 9. Completion Check & Experimental Insights

### 1. Why Poisoning Occurs Before Model Training
Poisoning attacks are **causative training-time attacks**. The learning algorithm searches for parameters $\theta^*$ that minimize the empirical training loss:
$$\theta^* = \arg\min_\theta \sum_{i \in D_{\text{train}}} L(f_\theta(x_i), y_i)$$
By flipping labels prior to optimization, the adversary directly shifts the empirical risk landscape, forcing the decision boundary to accommodate erroneous labels. Once the model is trained, the boundary is fixed, making training-time label poisoning impossible.

### 2. Experimental Invariants & Verification
- **Training features:** Untouched ($X_{\text{train}}$ hash is identical).
- **Test set isolation:** Both $X_{\text{test}}$ and $y_{\text{test}}$ are completely clean and identical to baseline.
- **Exact poison rate:** Exactly 184 labels were flipped ($184 / 3680 = 0.0500$).

### 3. Model Observations
- **Logistic Regression:** Spam recall drops significantly from **89.81% to 86.50% (-3.31%)**, showing linear classifiers are vulnerable to label noise.
- **Decision Tree:** Exhibits slight variance due to single-tree split instability.
- **Random Forest:** Retains strong stability (**90.63% recall**), highlighting the inherent robustness of ensemble bagging.